In [30]:
%pip install "nbformat>=4.2.0" nbformat ipython

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\RIDDHIMA PATRA\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [31]:
%pip install plotly

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\RIDDHIMA PATRA\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [32]:
%pip install "torch>=2.0.0" "torch_geometric>=2.3.0" "networkx>=3.0" "matplotlib>=3.7.0" "streamlit>=1.25.0"

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\RIDDHIMA PATRA\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Abstract
Traditional supply chain simulations often rely on deterministic, unconstrained networks. This notebook engineers a realistic spatial logistics graph where nodes possess physical coordinates, hierarchical roles (Fulfillment Hubs vs. Delivery Spokes), and dynamic capacity constraints. 

To predict non-linear cascading delays, we deploy a **Multi-Head Graph Attention Network (GAT)**. The model learns to "pay attention" to routing bottlenecks, successfully outperforming deterministic baseline heuristics.

In [33]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv # UPGRADED!
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import random
from math import radians, cos, sin, asin, sqrt # ADDED for Haversine distances
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
print("Libraries imported successfully.")

Libraries imported successfully.


## 1. Generating a Geographically Constrained Network
Real warehouses don't connect randomly. We generate a spatial graph where $N=30$ nodes are placed on a 2D coordinate grid. 
* Nodes only connect if the Euclidean distance between them is below a threshold (representing highway drive times).
* **Hubs (Fulfillment Centers):** 5 nodes.
* **Spokes (Delivery Stations):** 25 nodes.
* **Features:** Each node is assigned a `Capacity_Utilization` metric (40% to 95%).

## 2. Stochastic Data Generation
A simple math formula doesn't require ML. Here, delay propagation is non-linear. 
If Node A breaks down, the delay spreads to Node B based on:
1. Node B's current capacity (A 95% full warehouse bottlenecks instantly; a 40% full one absorbs the delay).
2. Random real-world noise (traffic variability).

In [34]:
hubs_data = [
    {"city": "Delhi", "lat": 28.4444, "lon": 76.8361, "type": 1.0}, {"city": "Mumbai", "lat": 19.3002, "lon": 73.0588, "type": 1.0},
    {"city": "Bengaluru", "lat": 12.8160, "lon": 77.6830, "type": 1.0}, {"city": "Chennai", "lat": 12.9734, "lon": 79.9482, "type": 1.0},
    {"city": "Kolkata", "lat": 22.6841, "lon": 88.2974, "type": 1.0}, {"city": "Durgapur", "lat": 23.5135, "lon": 87.3591, "type": 0.0},
    {"city": "Ahmedabad", "lat": 22.9859, "lon": 72.3831, "type": 0.0}, {"city": "Pune", "lat": 18.7501, "lon": 73.8340, "type": 0.0},
    {"city": "Hyderabad", "lat": 17.5284, "lon": 78.1904, "type": 0.0}, {"city": "Jaipur", "lat": 26.8144, "lon": 75.5414, "type": 0.0},
    {"city": "Lucknow", "lat": 26.7725, "lon": 80.8856, "type": 0.0}, {"city": "Kanpur", "lat": 26.4526, "lon": 80.3168, "type": 0.0},
    {"city": "Nagpur", "lat": 21.0360, "lon": 79.0354, "type": 0.0}, {"city": "Indore", "lat": 22.6145, "lon": 75.6796, "type": 0.0},
    {"city": "Bhopal", "lat": 23.0943, "lon": 77.5348, "type": 0.0}, {"city": "Patna", "lat": 25.5132, "lon": 85.2917, "type": 0.0},
    {"city": "Ludhiana", "lat": 30.8315, "lon": 75.9839, "type": 0.0}, {"city": "Agra", "lat": 27.2096, "lon": 77.9427, "type": 0.0},
    {"city": "Varanasi", "lat": 25.2690, "lon": 83.0335, "type": 0.0}, {"city": "Surat", "lat": 21.0772, "lon": 72.9691, "type": 0.0},
    {"city": "Kochi", "lat": 10.0468, "lon": 76.3197, "type": 0.0}, {"city": "Coimbatore", "lat": 11.0264, "lon": 77.1245, "type": 0.0},
    {"city": "Madurai", "lat": 9.8407, "lon": 79.9882, "type": 0.0}, {"city": "Guwahati", "lat": 26.1852, "lon": 91.6841, "type": 0.0},
    {"city": "Bhubaneswar", "lat": 20.1802, "lon": 85.6205, "type": 0.0}, {"city": "Dehradun", "lat": 30.3664, "lon": 77.8596, "type": 0.0},
    {"city": "Ranchi", "lat": 23.3320, "lon": 85.3670, "type": 0.0}, {"city": "Raipur", "lat": 21.2981, "lon": 81.5975, "type": 0.0},
    {"city": "Chandigarh", "lat": 30.6425, "lon": 76.8173, "type": 0.0}, {"city": "Visakhapatnam", "lat": 17.6908, "lon": 83.1645, "type": 0.0}
]

def haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    return 2 * asin(sqrt(sin((lat2 - lat1)/2)**2 + cos(lat1) * cos(lat2) * sin((lon2 - lon1)/2)**2)) * 6371

num_nodes = len(hubs_data)
G = nx.Graph()
pos = {}

for i, data in enumerate(hubs_data):
    pos[i] = (data['lon'], data['lat']) 
    capacity = np.random.uniform(0.50, 0.95) if data['type'] == 1.0 else np.random.uniform(0.30, 0.70)
    history_score = np.random.uniform(0.0, 0.2) if data['type'] == 1.0 else np.random.uniform(0.0, 0.5)
    G.add_node(i, pos=pos[i], node_type=data['type'], capacity=capacity, history=history_score, city=data['city'])

for i in range(num_nodes):
    for j in range(i + 1, num_nodes):
        dist_km = haversine(hubs_data[i]['lon'], hubs_data[i]['lat'], hubs_data[j]['lon'], hubs_data[j]['lat'])
        if dist_km < 600:
            traffic = np.random.uniform(0.8, 1.2)
            edge_weight = (dist_km / 600.0) * traffic 
            G.add_edge(i, j, weight=edge_weight)

# Ensure no isolated nodes
for i in range(num_nodes):
    if G.degree(i) == 0:
        distances = [haversine(hubs_data[i]['lon'], hubs_data[i]['lat'], hubs_data[j]['lon'], hubs_data[j]['lat']) if i != j else float('inf') for j in range(num_nodes)]
        nearest = np.argmin(distances)
        G.add_edge(i, nearest, weight=distances[nearest]/600.0)

edges = list(G.edges)
edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
edge_index = torch.cat([edge_index, edge_index[[1, 0]]], dim=1)

# Extract Edge Weights for PyTorch (CRITICAL FOR GATv2)
edge_weights = [G[u][v]['weight'] for u, v in edge_index.t().tolist()]
edge_attr = torch.tensor(edge_weights, dtype=torch.float32).view(-1, 1)

print("Real India Graph with Edge Weights created!")

Real India Graph with Edge Weights created!


In [39]:
import plotly.graph_objects as go

print("Generating Interactive Plotly Map...")

# 1. Extract Edge Coordinates
edge_lon = []
edge_lat = []
for edge in G.edges():
    lon0, lat0 = G.nodes[edge[0]]['pos']
    lon1, lat1 = G.nodes[edge[1]]['pos']
    edge_lon.extend([lon0, lon1, None])
    edge_lat.extend([lat0, lat1, None])

edge_trace = go.Scattergeo(
    lon=edge_lon, lat=edge_lat,
    mode='lines',
    line=dict(width=1, color='rgba(150, 150, 150, 0.5)'),
    hoverinfo='none' # Do not show tooltip for lines
)

# 2. Extract Node Coordinates and Features
node_lon = []
node_lat = []
hover_texts = []
node_colors = []
node_sizes = []

for node in G.nodes():
    lon, lat = G.nodes[node]['pos']
    node_lon.append(lon)
    node_lat.append(lat)
    
    is_hub = G.nodes[node]['node_type'] == 1.0
    city = G.nodes[node]['city']
    capacity = G.nodes[node]['capacity'] * 100
    
    # Hover text formatting
    hover_texts.append(f"<b>{'⭐ HUB' if is_hub else '📦 SPOKE'}: {city}</b><br>Capacity Util: {capacity:.1f}%")
    node_colors.append('#ff9900' if is_hub else '#232f3e')
    node_sizes.append(18 if is_hub else 8)

node_trace = go.Scattergeo(
    lon=node_lon, lat=node_lat,
    mode='markers+text',
    hoverinfo='text',
    hovertext=hover_texts,
    text=[G.nodes[n]['city'].split(" ")[0] if G.nodes[n]['node_type'] == 1.0 else "" for n in G.nodes()], # Only label hubs
    textposition="bottom center",
    marker=dict(
        size=node_sizes,
        color=node_colors,
        line=dict(width=1.5, color='white')
    )
)

# 3. Build the Map Figure
fig = go.Figure(data=[edge_trace, node_trace],
             layout=go.Layout(
                 title=dict(text='<b>Interactive India Logistics Network (600km Radius)</b>', font=dict(size=16)),
                 showlegend=False,
                 height=700,
                 margin=dict(l=0, r=0, t=40, b=0),
                 geo=dict(
                     scope='asia',
                     resolution=50,
                     # Lock the map strictly to India's bounding box
                     lonaxis=dict(range=[67, 98]),
                     lataxis=dict(range=[7, 36]),
                     showland=True,
                     landcolor="rgb(240, 240, 240)",
                     countrycolor="rgb(204, 204, 204)",
                     coastlinecolor="rgb(204, 204, 204)",
                     projection_type='mercator'
                 )
             ))

fig.show(renderer="browser")

Generating Interactive Plotly Map...


In [36]:
num_samples = 1000 
X_data, Y_data = [], []

capacities = torch.tensor([G.nodes[i]['capacity'] for i in range(num_nodes)], dtype=torch.float32)
types = torch.tensor([G.nodes[i]['node_type'] for i in range(num_nodes)], dtype=torch.float32)
histories = torch.tensor([G.nodes[i]['history'] for i in range(num_nodes)], dtype=torch.float32)

for _ in range(num_samples):
    initial_delay = torch.zeros(num_nodes)
    broken_nodes = random.sample(range(num_nodes), k=random.randint(1, 2))
    for node in broken_nodes: initial_delay[node] = 1.0 
        
    # INPUT: Now has 4 Features per node!
    x = torch.stack([initial_delay, capacities, types, histories], dim=1)
    
    # PHYSICS: 3-Hop Simulation (T+3)
    delays = initial_delay.clone().numpy()
    for step in range(3):
        new_delays = delays.copy()
        for u, v in G.edges():
            efficiency = 1.0 / (G[u][v]['weight'] + 0.1) # Faster propagation on low-weight edges
            if delays[u] > 0.1: new_delays[v] = min(1.0, new_delays[v] + delays[u] * (capacities[v].item()**2) * efficiency * 0.4)
            if delays[v] > 0.1: new_delays[u] = min(1.0, new_delays[u] + delays[v] * (capacities[u].item()**2) * efficiency * 0.4)
        delays = new_delays
        
    y = torch.tensor(delays, dtype=torch.float32).view(-1, 1)
    X_data.append(x)
    Y_data.append(y)

print(f"Generated {len(X_data)} T+3 multi-hop scenario matrices.")

Generated 1000 T+3 multi-hop scenario matrices.


## 3. The Graph Attention Network (GAT) Architecture
Instead of a basic GCN that averages neighbor features blindly, we use a **Multi-Head Graph Attention Network (`GATConv`)**. The GAT mathematically learns *which* neighboring roads and warehouses are the most critical, assigning them higher attention weights during the message-passing phase.

In [37]:
class ExplainableLogisticsGAT(torch.nn.Module):
    def __init__(self):
        super(ExplainableLogisticsGAT, self).__init__()
        # 4 input features, edge_dim=1 for weights
        self.gat1 = GATv2Conv(in_channels=4, out_channels=16, heads=4, edge_dim=1, concat=True)
        self.gat2 = GATv2Conv(in_channels=64, out_channels=1, heads=1, edge_dim=1, concat=False)

    def forward(self, x, edge_index, edge_attr, return_attention=False):
        # Extract alpha (attention weights) for explainability
        x, (edge_index1, alpha1) = self.gat1(x, edge_index, edge_attr, return_attention_weights=True)
        x = F.elu(x)
        x, (edge_index2, alpha2) = self.gat2(x, edge_index, edge_attr, return_attention_weights=True)
        out = torch.sigmoid(x)
        
        if return_attention: return out, alpha1
        return out

model = ExplainableLogisticsGAT()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
criterion = torch.nn.MSELoss()
print(model)

ExplainableLogisticsGAT(
  (gat1): GATv2Conv(4, 16, heads=4)
  (gat2): GATv2Conv(64, 1, heads=1)
)


In [38]:
epochs = 250
model.train()

print("Training Explainable Edge-Weighted GATv2...")
for epoch in range(epochs):
    total_loss = 0
    for i in range(num_samples):
        optimizer.zero_grad()
        # CRITICAL: Pass edge_attr into the model!
        out = model(X_data[i], edge_index, edge_attr)
        loss = criterion(out, Y_data[i])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if epoch % 50 == 0 or epoch == epochs - 1:
        print(f'Epoch {epoch:>3} | GAT MSE Loss: {total_loss/num_samples:.5f}')

# --- SAVE ARTIFACTS ---
torch.save(model.state_dict(), 'gat_logistics_model.pth')
torch.save({'edge_index': edge_index, 'edge_attr': edge_attr}, 'graph_edges.pth') # Saved edge_attr
torch.save({'capacities': capacities, 'types': types, 'histories': histories, 'pos': pos}, 'graph_meta.pth') # Saved histories

print("✅ Training complete! Artifacts saved for Streamlit deployment.")

Training Explainable Edge-Weighted GATv2...
Epoch   0 | GAT MSE Loss: 0.03297
Epoch  50 | GAT MSE Loss: 0.00305
Epoch 100 | GAT MSE Loss: 0.00247
Epoch 150 | GAT MSE Loss: 0.00223
Epoch 200 | GAT MSE Loss: 0.00200
Epoch 249 | GAT MSE Loss: 0.00193
✅ Training complete! Artifacts saved for Streamlit deployment.
